# Regressão Logística com Validação de Premissas - Sklearn

## Bibliotecas e Configuração

In [ ]:
# Dependências
import sys
#!{sys.executable} -m pip install --disable-pip-version-check -r ../requirements.txt -q
print('Bibliotecas instaladas')

In [ ]:
# Dependências
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Manipulação dos dados
import pandas as pd
import numpy as np
from datetime import datetime
import pickle
import os

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Diretórios
from configs.paths import *
from configs.function_basic import *
from configs.function_others import *

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

# Métricas e validação
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score,precision_recall_curve,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Avisos
import warnings
warnings.filterwarnings('ignore')

# Configuração
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print('✅ Ambiente Configurado')

## Parâmetros Globais

In [ ]:
# Definições
TARGET = 'FPD'
RANDOM_STATE = 42
IV = 0.04
THRESHOLD = 0.5
C = 0.1
DATA_EXECUCAO = datetime.now().strftime('%d-%m-%Y')
VERSAO = 'V1-4 - RL(Sklearn COM Premissas)'

- V1 - Ajustes das premissas
- V1-2 - Ajustes das premissas + normalização
- V1-3 - Ajustes das premissas + normalização + VIF(0.02)
- V1-4 - Ajustes das premissas + normalização + VIF(0.04)

## Carregamento dos Dados

In [ ]:
# Carregar dados CORRIGIDOS conforme solicitado
train = pd.read_csv(PROCESSED_DIR / 'abt01_train.csv')
test = pd.read_csv(PROCESSED_DIR / 'abt01_test.csv')

print(f'📊 Treino: {train.shape}')
print(f'📊 Teste: {test.shape}')
print(f'\n🎯 Distribuição do Target (Treino):')
print(f"  Treino: {(train[TARGET].value_counts(normalize=True) * 100).round(2)}")

## Preparação dos Dados

In [ ]:
# Backup dos dados originais
train_01 = train.copy()
test_01 = test.copy()

# lista de vars para retirar dos tratamentos
ignore_cols = ['SAFRA']

# Aplicando no treino
train_01 = train_01.drop(columns=ignore_cols)
test_01 = test_01.drop(columns=ignore_cols)

In [ ]:
# Separar features e target
X_train = train_01.drop(TARGET, axis=1)
y_train = train_01[TARGET]

X_test = test_01.drop(TARGET, axis=1)
y_test = test_01[TARGET]

print(f'X_train: {X_train.shape}')
print(f'X_test: {X_test.shape}')

# Garantir mesmas features em treino e teste
features_common = X_train.columns.intersection(X_test.columns)
X_train = X_train[features_common]
X_test = X_test[features_common]

print(f'\n✅ Features alinhadas: {len(features_common)}')

## Normalização das Features

In [ ]:
'''
# Dados não normalizados
X_train_scaled = X_train
X_test_scaled = X_test
'''

In [ ]:

# Normalizar as features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Converter para DataFrame para manter nomes das features
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print('✅ Features Normalizadas')


## Validação das Premissas da Regressão Logística

### Multicolinearidade (VIF)

In [ ]:

# Calcular VIF para cada feature
iv_df = iv_table(train, TARGET)
iv_df.head(100)


In [ ]:
# treina usando apenas as features selecionadas
selected_features = iv_df[iv_df.IV > IV].Variável.tolist()
X_train_scaled = X_train_scaled[selected_features]

In [ ]:
# Salvar a lista em um arquivo .pkl
artifact_path = Path(ARTIFACT_DIR) / 'selected_features_IV.pkl'

with open(artifact_path, 'wb') as f:
    pickle.dump(selected_features, f)

### Linearidade no Logit (Log-Odds)

In [ ]:
# Listar as colunas selecionadas no VIF
selected_features = X_train_scaled.columns.tolist()


In [ ]:
# Calcular R² dos log-odds e filtrar variáveis correlacionadas
r2_df = calculate_r2_for_logodds(
    X_train_scaled.assign(**{TARGET: y_train}),
    selected_features,
    TARGET,
    threshold=0.85
)

In [ ]:
# Identificar variáveis que precisam ser categorizadas
vars_to_bin = r2_df[r2_df['Feat Eng'] == 'Categorizar'].Variable.tolist()

### Binning supervisionado (Decision Tree)

In [ ]:
# objetivo: montar dataset final com bins + safra + target
df_safra = X_train_scaled[vars_to_bin].copy()
df_safra["SAFRA"] = train.loc[X_train_scaled.index, "SAFRA"]
df_safra[TARGET] = y_train.loc[X_train_scaled.index]

In [ ]:
# Gerar tabela de metadados
metadados = dataset_info_table(df_safra)

In [ ]:
# filtra variáveis
df_high_card = metadados[(metadados['Cardinalidade'] > 1000)]['Feature'].tolist()
print(df_high_card)

In [ ]:
# propósito: agrupar variáveis de alta cardinalidade via árvore rasa supervisionada
modelos_arvores_high = {}

for col in df_high_card:

    #print(f'Processando: {col}')

    # X com limpeza pesada
    X = df_safra[[col]].replace([np.inf, -np.inf], np.nan).fillna(-999)

    # y alinhado pelo índice
    y =df_safra[TARGET]

    # sanity check
    if len(X) != len(y):
        raise ValueError(f'Desalinhamento em {col}')

    # treina árvore rasa
    tree = DecisionTreeClassifier(
        max_leaf_nodes=10,
        min_samples_leaf=100,   # evita folha lixo
        random_state=42
    )
    tree.fit(X, y)

    # aplica agrupamento
    df_safra[f'{col}_agrupada'] = tree.apply(X)

    # guarda modelo
    modelos_arvores_high[col] = tree

In [ ]:
# Filtrar variáveis de cardinalidade média (100 a 1000)
df_med_card = metadados[(metadados['Cardinalidade'] >= 100) & (metadados['Cardinalidade'] <= 1000)]['Feature'].tolist()
print(df_med_card)

In [ ]:
# propósito: agrupar variáveis de média cardinalidade via árvore rasa
modelos_arvores_med = {}

for col in df_med_card:

    #print(f'Processando: {col}')

    X = df_safra[[col]].replace([np.inf, -np.inf], np.nan).fillna(-999)
    y = df_safra[TARGET]

    if len(X) != len(y):
        raise ValueError(f'Desalinhamento em {col}')

    tree = DecisionTreeClassifier(
        max_leaf_nodes=6,        # menos grupos
        min_samples_leaf=200,    # mais estabilidade
        random_state=42
    )

    tree.fit(X, y)

    df_safra[f'{col}_agrupada'] = tree.apply(X)
    modelos_arvores_med[col] = tree


In [ ]:
# Filtrar variáveis de cardinalidade Baixa (11 a 99)
df_low_card = metadados[(metadados['Cardinalidade'] >= 11) & (metadados['Cardinalidade'] <= 99)]['Feature'].tolist()
print(df_low_card)

In [ ]:
# propósito: agrupar variáveis de média cardinalidade via árvore rasa
modelos_arvores_low = {}

for col in df_low_card:

    #print(f'Processando: {col}')

    X = df_safra[[col]].replace([np.inf, -np.inf], np.nan).fillna(-999)
    y = df_safra[TARGET]

    if len(X) != len(y):
        raise ValueError(f'Desalinhamento em {col}')

    tree = DecisionTreeClassifier(
        max_leaf_nodes=3,        # pouquíssimos grupos
        min_samples_leaf=500,    # estabilidade > granularidade
        random_state=42
    )

    tree.fit(X, y)

    df_safra[f'{col}_agrupada'] = tree.apply(X)
    modelos_arvores_low[col] = tree

In [ ]:
# Unir colunas de média e alta cardinalidade
cols_card = list(set(df_high_card + df_med_card + df_low_card))
# Remover colunas de média e alta cardinalidade
df_safra = df_safra.drop(columns=cols_card)

In [ ]:
# Salvar a lista em um arquivo .pkl
artifact_path = Path(ARTIFACT_DIR) / 'cols_drop.pkl'

with open(artifact_path, 'wb') as f:
    pickle.dump(cols_card, f)

In [ ]:
# Salvar modelos de binning supervisionado

artefato_binning = {
    'high_card': modelos_arvores_high,
    'med_card': modelos_arvores_med,
    'low_card' :modelos_arvores_low
}

artifact_path = Path(ARTIFACT_DIR) / 'binning_arvores.pkl'

with open(artifact_path, 'wb') as f:
    pickle.dump(artefato_binning, f)

In [ ]:
'''
# Gerar gráficos por safra apenas para variáveis agrupadas
for col in df_safra.columns:
    if col.endswith('_agrupada'):
        plot_by_safra(df_safra, TARGET, col, "SAFRA")
'''

In [ ]:
## Pós-processamento de folhas específicas após binning supervisionado
## Todos
#df_safra.loc[df_safra['var_04'].isin([1, 2, 3, 4]),'var_04'] = 5
#df_safra.loc[df_safra['var_29_agrupada'].isin([3]),'var_29_agrupada'] = 4
#df_safra.loc[df_safra['var_51_agrupada'].isin([6]),'var_51_agrupada'] = 9
#df_safra.loc[df_safra['var_03_agrupada'].isin([5, 6, 8, 9]),'var_03_agrupada'] = 10
#df_safra.loc[df_safra['QTD_REGISTROS_LAG_4_agrupada'].isin([3]),'QTD_REGISTROS_LAG_4_agrupada'] = 10
#df_safra.loc[df_safra['QTD_REGISTROS_LAG_5_agrupada'].isin([3]),'QTD_REGISTROS_LAG_5_agrupada'] = 6
#df_safra.loc[df_safra['QTD_REGISTROS_LAG_3_agrupada'].isin([1]),'QTD_REGISTROS_LAG_3_agrupada'] = 7
#df_safra.loc[df_safra['QTD_REGISTROS_LAG_2_agrupada'].isin([1]),'QTD_REGISTROS_LAG_2_agrupada'] = 8

## VIF (0.02)
#df_safra.loc[df_safra['REGIAO_POSTAL'].isin([4, 5]),'REGIAO_POSTAL'] = 6
#df_safra.loc[df_safra['REGIAO_POSTAL'].isin([0, 1, 2, 3, 7, 8]),'REGIAO_POSTAL'] = 9
#df_safra.loc[df_safra['var_82_agrupada'].isin([6]),'var_82_agrupada'] = 10
#df_safra.loc[df_safra['var_82_agrupada'].isin([5]),'var_82_agrupada'] = 9
#df_safra.loc[df_safra['SUB_REGIAO_POSTAL_agrupada'].isin([1]),'SUB_REGIAO_POSTAL_agrupada'] = 4
#df_safra.loc[df_safra['REGIAO_POSTAL_TXT_enc_agrupada'].isin([1]),'REGIAO_POSTAL_TXT_enc_agrupada'] = 3
#df_safra.loc[df_safra['var_58_agrupada'].isin([1]),'var_58_agrupada'] = 10
#df_safra.loc[df_safra['var_43_agrupada'].isin([4]),'var_43_agrupada'] = 7
#df_safra.loc[df_safra['var_43_agrupada'].isin([3]),'var_43_agrupada'] = 5
#df_safra.loc[df_safra['var_42_agrupada'].isin([4]),'var_42_agrupada'] = 9
#df_safra.loc[df_safra['var_42_agrupada'].isin([3, 7, 8]),'var_42_agrupada'] = 10
#df_safra.loc[df_safra['CEP_3_digitos_agrupada'].isin([4]),'CEP_3_digitos_agrupada'] = 7
#df_safra.loc[df_safra['QTD_REGISTROS_ULT_6_SAFRAS_agrupada'].isin([3, 6]),'QTD_REGISTROS_ULT_6_SAFRAS_agrupada'] = 7
#df_safra.loc[df_safra['QTD_REGISTROS_ULT_3_SAFRAS_agrupada'].isin([7]),'QTD_REGISTROS_ULT_3_SAFRAS_agrupada'] = 9
#df_safra.loc[df_safra['QTD_REGISTROS_ULT_1_SAFRAS_agrupada'].isin([5, 6]),'QTD_REGISTROS_ULT_1_SAFRAS_agrupada'] = 7
#df_safra.loc[df_safra['QTD_REGISTROS_ULT_1_SAFRAS_agrupada'].isin([8]),'QTD_REGISTROS_ULT_1_SAFRAS_agrupada'] = 9
#df_safra.loc[df_safra['QTD_REGISTROS_LAG_1_agrupada'].isin([1]),'QTD_REGISTROS_LAG_1_agrupada'] = 7
#df_safra.loc[df_safra['QTD_REGISTROS_agrupada'].isin([6]),'QTD_REGISTROS_agrupada'] = 10

## VIF (0.03)
df_safra.loc[df_safra['var_05'].isin([1, 2]),'var_05'] = 10
df_safra.loc[df_safra['var_05'].isin([3, 4, 5, 6, 7, 8]),'var_05'] = 9
df_safra.loc[df_safra['var_72_agrupada'].isin([2]),'var_72_agrupada'] = 3
df_safra.loc[df_safra['var_30_agrupada'].isin([2]),'var_30_agrupada'] = 5
df_safra.loc[df_safra['var_28_agrupada'].isin([5]),'var_28_agrupada'] = 9
df_safra.loc[df_safra['VALOR_SOS_ULT_3_SAFRAS_agrupada'].isin([1, 7, 8, 9]),'VALOR_SOS_ULT_3_SAFRAS_agrupada'] = 10

In [ ]:
# atualizar lista cols_drop com variaveis transformadas manualmente
artifact_path = Path(ARTIFACT_DIR) / 'cols_drop.pkl'

with open(artifact_path, 'rb') as f:
    cols_drop = pickle.load(f)

cols_drop.extend(['var_04', 'var_05', 'REGIAO_POSTAL'])

# remover duplicados por segurança
cols_drop_bin = list(set(cols_drop))

# Salvar a lista em um arquivo .pkl
artifact_path = Path(ARTIFACT_DIR) / 'cols_drop_bin.pkl'

with open(artifact_path, 'wb') as f:
    pickle.dump(cols_drop_bin, f)

In [ ]:
# remover colunas não desejadas do X_train_scaled
artifact_path = Path(ARTIFACT_DIR) / 'cols_drop_bin.pkl'

with open(artifact_path, 'rb') as f:
    cols_drop_bin = pickle.load(f)
    
X_train_scaled = X_train_scaled.drop(columns=cols_drop_bin, errors="ignore")

## Ajustando o Dataset

In [ ]:
# renomear multiplas colunas
df_safra = df_safra.rename(columns={
    "var_04": "var_04_agrupada",
    "var_05": "var_05_agrupada",
    "REGIAO_POSTAL": "REGIAO_POSTAL_agrupada"

})
# lista de vars para retirar dos tratamentos
ignore_cols = ['SAFRA', 'FPD']

# Aplicando 
df_safra = df_safra.drop(columns=ignore_cols)

In [ ]:
# checar se a quantidade de linhas bate
print("X_train_scaled:", X_train_scaled.shape[0])
print("df_safra:", df_safra.shape[0])

In [ ]:
# checar se os índices são exatamente os mesmos
X_train_scaled.index.equals(df_safra.index)

In [ ]:
# alinhar df_safra ao X_train_scaled
df_safra_aligned = df_safra.loc[X_train_scaled.index]

# adicionar variáveis agrupadas ao treino
X_train_scaled = pd.concat(
    [X_train_scaled, df_safra_aligned],
    axis=1
)

In [ ]:
'''
# Validaçã
X_train_final["SAFRA"] = train.loc[X_train_scaled.index, "SAFRA"]
X_train_final[TARGET] = train.loc[X_train_scaled.index, TARGET]

plot_by_safra(X_train_final, TARGET, 'var_43', "SAFRA")
'''

### Preparar dados do Teste

In [ ]:
# carregar features por IV e aplicar no teste apenas se existirem
artifact_path = Path(ARTIFACT_DIR) / 'selected_features_IV.pkl'

# carregar lista de features
selected_features_IV = []
if artifact_path.exists() and artifact_path.stat().st_size > 0:
    with open(artifact_path, 'rb') as f:
        selected_features_IV = pickle.load(f)

# aplicar somente se a lista nao estiver vazia
if selected_features_IV:
    # garantir que as colunas existem no teste
    cols_validas = [c for c in selected_features_IV if c in X_test_scaled.columns]

    if cols_validas:
        X_test_scaled = X_test_scaled[cols_validas].copy()


In [ ]:
# Aplicar binning supervisionado no novo dataset
with open(Path(ARTIFACT_DIR) / 'binning_arvores.pkl', 'rb') as f:
    artefato_binning = pickle.load(f)

with open(Path(ARTIFACT_DIR) / 'cols_drop.pkl', 'rb') as f:
    cols_card = pickle.load(f)

# aplica árvores
for grupo in ['high_card', 'med_card', 'low_card']:
    for col, tree in artefato_binning[grupo].items():
        X = X_test_scaled[[col]].replace([np.inf, -np.inf], np.nan).fillna(-999)
        X_test_scaled[f'{col}_agrupada'] = tree.apply(X)

# remove colunas originais
X_test_scaled = X_test_scaled.drop(columns=cols_card)

In [ ]:
## Pós-processamento de folhas específicas após binning supervisionado

## Todos
#X_test_scaled.loc[X_test_scaled['var_04'].isin([1, 2, 3, 4]),'var_04'] = 5
#X_test_scaled.loc[X_test_scaled['var_29_agrupada'].isin([3]),'var_29_agrupada'] = 4
#X_test_scaled.loc[X_test_scaled['var_51_agrupada'].isin([6]),'var_51_agrupada'] = 9
#X_test_scaled.loc[X_test_scaled['var_03_agrupada'].isin([5, 6, 8, 9]),'var_03_agrupada'] = 10
#X_test_scaled.loc[X_test_scaled['QTD_REGISTROS_LAG_4_agrupada'].isin([3]),'QTD_REGISTROS_LAG_4_agrupada'] = 10
#X_test_scaled.loc[X_test_scaled['QTD_REGISTROS_LAG_5_agrupada'].isin([3]),'QTD_REGISTROS_LAG_5_agrupada'] = 6
#X_test_scaled.loc[X_test_scaled['QTD_REGISTROS_LAG_3_agrupada'].isin([1]),'QTD_REGISTROS_LAG_3_agrupada'] = 7
#X_test_scaled.loc[X_test_scaled['QTD_REGISTROS_LAG_2_agrupada'].isin([1]),'QTD_REGISTROS_LAG_2_agrupada'] = 8

## VIF (0.02)
#X_test_scaled.loc[X_test_scaled['REGIAO_POSTAL'].isin([4, 5]),'REGIAO_POSTAL'] = 6
#X_test_scaled.loc[X_test_scaled['REGIAO_POSTAL'].isin([0, 1, 2, 3, 7, 8]),'REGIAO_POSTAL'] = 9
#X_test_scaled.loc[X_test_scaled['var_82_agrupada'].isin([6]),'var_82_agrupada'] = 10
#X_test_scaled.loc[X_test_scaled['var_82_agrupada'].isin([5]),'var_82_agrupada'] = 9
#X_test_scaled.loc[X_test_scaled['SUB_REGIAO_POSTAL_agrupada'].isin([1]),'SUB_REGIAO_POSTAL_agrupada'] = 4
#X_test_scaled.loc[X_test_scaled['REGIAO_POSTAL_TXT_enc_agrupada'].isin([1]),'REGIAO_POSTAL_TXT_enc_agrupada'] = 3
#X_test_scaled.loc[X_test_scaled['var_58_agrupada'].isin([1]),'var_58_agrupada'] = 10
#X_test_scaled.loc[X_test_scaled['var_43_agrupada'].isin([4]),'var_43_agrupada'] = 7
#X_test_scaled.loc[X_test_scaled['var_43_agrupada'].isin([3]),'var_43_agrupada'] = 5
#X_test_scaled.loc[X_test_scaled['var_42_agrupada'].isin([4]),'var_42_agrupada'] = 9
#X_test_scaled.loc[X_test_scaled['var_42_agrupada'].isin([3, 7, 8]),'var_42_agrupada'] = 10
#X_test_scaled.loc[X_test_scaled['CEP_3_digitos_agrupada'].isin([4]),'CEP_3_digitos_agrupada'] = 7
#X_test_scaled.loc[X_test_scaled['QTD_REGISTROS_ULT_6_SAFRAS_agrupada'].isin([3, 6]),'QTD_REGISTROS_ULT_6_SAFRAS_agrupada'] = 7
#X_test_scaled.loc[X_test_scaled['QTD_REGISTROS_ULT_3_SAFRAS_agrupada'].isin([7]),'QTD_REGISTROS_ULT_3_SAFRAS_agrupada'] = 9
#X_test_scaled.loc[X_test_scaled['QTD_REGISTROS_ULT_1_SAFRAS_agrupada'].isin([5, 6]),'QTD_REGISTROS_ULT_1_SAFRAS_agrupada'] = 7
#X_test_scaled.loc[X_test_scaled['QTD_REGISTROS_ULT_1_SAFRAS_agrupada'].isin([8]),'QTD_REGISTROS_ULT_1_SAFRAS_agrupada'] = 9
#X_test_scaled.loc[X_test_scaled['QTD_REGISTROS_LAG_1_agrupada'].isin([1]),'QTD_REGISTROS_LAG_1_agrupada'] = 7
#X_test_scaled.loc[X_test_scaled['QTD_REGISTROS_agrupada'].isin([6]),'QTD_REGISTROS_agrupada'] = 10

## VIF (0.03)
X_test_scaled.loc[X_test_scaled['var_05'].isin([1, 2]),'var_05'] = 10
X_test_scaled.loc[X_test_scaled['var_05'].isin([3, 4, 5, 6, 7, 8]),'var_05'] = 9
X_test_scaled.loc[X_test_scaled['var_72_agrupada'].isin([2]),'var_72_agrupada'] = 3
X_test_scaled.loc[X_test_scaled['var_30_agrupada'].isin([2]),'var_30_agrupada'] = 5
X_test_scaled.loc[X_test_scaled['var_28_agrupada'].isin([5]),'var_28_agrupada'] = 9
X_test_scaled.loc[X_test_scaled['VALOR_SOS_ULT_3_SAFRAS_agrupada'].isin([1, 7, 8, 9]),'VALOR_SOS_ULT_3_SAFRAS_agrupada'] = 10


In [ ]:
# renomear multiplas colunas
X_test_scaled = X_test_scaled.rename(columns={
    "var_04": "var_04_agrupada",
    "var_05": "var_05_agrupada",
    "REGIAO_POSTAL": "REGIAO_POSTAL_agrupada"
})

### Comparar colunas

In [ ]:
# Colunas do treino e teste
cols_train = set(X_train_scaled.columns)
cols_test = set(X_test_scaled.columns)

# Verifica se há diferença
print("Colunas diferentes entre treino e teste:", cols_train.symmetric_difference(cols_test))

### Garantir ordem das colunas igual ao treino

In [ ]:
# Alinha a ordem das colunas do teste com o treino para garantir consistência no modelo
X_test_scaled = X_test_scaled[X_train_scaled.columns]

## Treinamento do Modelo - Sklearn

In [ ]:
# treinar regressão logística com balanceamento automático

print('🤖 Treinando Regressão Logística (Sklearn) com class_weight=balanced...')

model = LogisticRegression(
    random_state=RANDOM_STATE,
    max_iter=1000,
    solver='lbfgs',
    class_weight='balanced',
    C=C,
    n_jobs=-1,
    verbose=0
)
model.fit(X_train_scaled, y_train)

print('✅ Modelo treinado com sucesso!')

print(f'\n📊 Informações do Modelo:')
print(f'Classes: {model.classes_}')
print(f'Coeficientes: {len(model.coef_[0])} features')
print(f'Intercepto: {model.intercept_[0]:.4f}')


## Importância das Features

In [ ]:
# Análise de coeficientes
feature_importance = pd.DataFrame({
    'Feature': X_train_scaled.columns,
    'Coeficiente': model.coef_[0]
})

feature_importance['Abs_Coef'] = np.abs(feature_importance['Coeficiente'])
feature_importance = feature_importance.sort_values('Abs_Coef', ascending=False)

print('\n📈 TOP 10 FEATURES MAIS IMPORTANTES:')
print(feature_importance.head(10)[['Feature', 'Coeficiente']].to_string(index=False))

## Predições no Conjunto de Teste

In [ ]:
# Fazer predições
print('🔮 Fazendo predições no conjunto de teste...')
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
y_pred = model.predict(X_test_scaled)

print(f'✅ Predições concluídas!')
print(f'Proporção de classe 1: {y_pred.mean():.4f}')

## Métricas de Avaliação

In [ ]:
# Calcular métricas
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print('\n📋 RELATÓRIO DE CLASSIFICAÇÃO:')
print(classification_report(y_test, y_pred, target_names=['Bom (0)', 'Mau (1)']))

## Matriz de Confusão

In [ ]:
# Matriz de confusão
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

print('\nConfusion Matrix:')
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Bom (0)', 'Mau (1)'])

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, values_format='d')
plt.title('Confusion Matrix – Regressão Logística (Sklearn) com Premissas')
plt.tight_layout()
plt.show()

## Salvamento das Métricas

In [ ]:
# Criar registro de métricas
metrics_row = {
    'Precision': precision,
    'Recall': recall,
    'F1_score': f1,
    'AUC': auc,
    'TP': int(tp),
    'TN': int(tn),
    'FP': int(fp),
    'FN': int(fn),
    'Versao_Modelo': VERSAO,
    'Data_Execucao': DATA_EXECUCAO,
}

df_metrics = pd.DataFrame([metrics_row])

print('\n📊 MÉTRICAS DO MODELO:')
print(df_metrics.to_string(index=False))

## Atualizar Arquivo de Métricas

In [ ]:
# Salvar/atualizar metricas
path_metrics = METRICS_DIR / 'model_metrics.csv'

if os.path.exists(path_metrics):
    df_hist = pd.read_csv(path_metrics)
    
    # Se versão já existe, atualiza
    if VERSAO in df_hist['Versao_Modelo'].values:
        df_hist.loc[df_hist['Versao_Modelo'] == VERSAO] = df_metrics.iloc[0].values
    else:
        df_hist = pd.concat([df_hist, df_metrics], ignore_index=True)
else:
    df_hist = df_metrics

df_hist.to_csv(path_metrics, index=False)
print(f'✅ Métricas salvas em: {path_metrics}')

# Mostrar histórico de versões
print('\n📜 HISTÓRICO DE VERSÕES:')
df_hist = pd.read_csv(path_metrics)
df_hist = df_hist.sort_values('Data_Execucao', ascending=False)
print(df_hist[['Versao_Modelo', 'Precision', 'Recall', 'F1_score', 'AUC', 'TN', 'TP', 'FP', 'FN', 'Data_Execucao']].to_string(index=False))